# 第13章（二）：文件组织与访问 File Organization and Access## Cambridge A-Level Computer Science (9618)---### 本节学习目标 Learning Objectives1. 理解三种**文件组织方式**：串行 (Serial)、顺序 (Sequential)、随机/直接 (Random/Direct)2. 理解三种**文件访问方式**：顺序访问 (Sequential Access)、随机访问 (Random Access)、索引顺序访问 (Indexed Sequential Access)3. 掌握**哈希算法 (Hashing Algorithms)** 在文件访问中的应用4. 理解**索引文件 (Index Files)** 和**多级索引 (Multi-level Indexes)** 的工作原理5. 能够根据实际场景选择合适的文件组织和访问方式---### 为什么需要理解文件组织？Why Understand File Organization?想象你在一个巨大的图书馆里找一本书：- **串行 (Serial)**：书随意堆放在地上，要找某本书只能一本一本翻——效率极低- **顺序 (Sequential)**：书按字母顺序排列在书架上，可以按顺序查找- **随机/直接 (Random)**：每本书有编号，输入编号直接定位到书架位置——最快！在计算机中，**文件就是数据的"图书馆"**，文件组织方式决定了数据如何存储和检索，直接影响程序的性能。> **Cambridge 考试重点**：你需要能够比较不同文件组织和访问方式的**优缺点**，并能根据场景推荐合适的方法。

In [ ]:
# 环境配置 Environment Setupimport matplotlib.pyplot as pltimport matplotlibimport matplotlib.patches as mpatchesimport numpy as npimport osimport structimport jsonimport timeimport tempfilefrom IPython.display import display, HTML, Markdownmatplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']matplotlib.rcParams['axes.unicode_minus'] = Falseplt.rcParams['figure.figsize'] = (12, 6)print('环境配置完成！Ready to explore file organization!')

---## 一、文件组织方式 File Organization Methods文件组织方式描述的是**数据在磁盘上如何物理排列**。这就像是决定图书馆里的书要怎么摆放。| 组织方式 | 英文 | 排序？ | 关键特征 ||---------|------|-------|----------|| 串行 | Serial | 无序 | 数据按**到达顺序**写入 || 顺序 | Sequential | 有序 | 数据按**关键字段 (key field)** 排序 || 随机/直接 | Random/Direct | 无固定序 | 通过**哈希函数**计算存储地址 |### 1.1 串行文件 Serial File**定义**：记录按照**写入的时间顺序**依次追加到文件末尾，没有任何排序。**生活类比**：想象你的微信聊天记录——消息按照发送时间依次排列，不会按内容或发送者排序。**特点**：- 写入非常快（直接追加到末尾）- 查找极慢（必须从头到尾遍历，平均需要检查 N/2 条记录）- 适合**日志文件 (log files)**、**临时数据收集**- 新记录总是添加到文件末尾**Cambridge 伪代码**：```OPENFILE "log.dat" FOR APPENDWRITEFILE "log.dat", LogRecordCLOSEFILE "log.dat"```

In [ ]:
# 演示：串行文件 - 数据按到达顺序存储# Demo: Serial File - data stored in arrival orderserial_log = []# 模拟事件按时间到达（注意：ID是无序的！）events = [    {"time": "09:15", "id": 47, "event": "用户登录 User Login"},    {"time": "09:16", "id": 12, "event": "文件上传 File Upload"},    {"time": "09:18", "id": 85, "event": "数据查询 Data Query"},    {"time": "09:20", "id": 3,  "event": "用户退出 User Logout"},    {"time": "09:22", "id": 56, "event": "密码修改 Password Change"},    {"time": "09:25", "id": 31, "event": "系统备份 System Backup"},]print("=" * 60)print("串行文件 (Serial File) - 按到达顺序存储")print("=" * 60)header = f"{'位置':<6}{'时间':<8}{'ID':<6}{'事件'}"print(header)print("-" * 60)for i, e in enumerate(events):    serial_log.append(e)    print(f"{i:<6}{e['time']:<8}{e['id']:<6}{e['event']}")print(f"\n注意：ID 是无序的 (47, 12, 85, 3, 56, 31)")print(f"   如果要查找 ID=3 的记录，需要遍历到第 4 条才能找到！")

### 1.2 顺序文件 Sequential File**定义**：记录按照某个**关键字段 (key field)** 的值**排序**后存储。**生活类比**：想象一本按字母顺序排列的电话簿——你知道 "Zhang" 一定在 "Li" 的后面。**特点**：- 查找比串行快（可以用**二分查找**，O(log N)）- 批量按顺序处理非常高效（如打印所有学生成绩单）- **插入和删除很慢！** 需要重写整个文件来维持排序- 适合**批量处理**场景（如月度工资单、年终报表）**关键概念 - 更新顺序文件**：顺序文件的更新通常使用 **"主文件 + 事务文件 -> 新主文件"** 的模式：1. 读取旧的**主文件 (Master File)**（已排序）2. 读取**事务文件 (Transaction File)**（包含新增、修改、删除的记录）3. **合并**两个文件，生成新的主文件```旧主文件 ---+            +---> 合并处理 ---> 新主文件事务文件 ---+```

In [ ]:
# 演示：顺序文件 - 按关键字段排序存储 + 二分查找# Demo: Sequential File - stored sorted by key field + Binary Searchsequential_records = [    {"id": 1001, "name": "Alice",  "score": 92},    {"id": 1002, "name": "Bob",    "score": 85},    {"id": 1005, "name": "Charlie","score": 78},    {"id": 1008, "name": "Diana",  "score": 95},    {"id": 1012, "name": "Eve",    "score": 88},    {"id": 1015, "name": "Frank",  "score": 71},    {"id": 1020, "name": "Grace",  "score": 93},    {"id": 1025, "name": "Hank",   "score": 82},]print("=" * 60)print("顺序文件 (Sequential File) - 按学号排序")print("=" * 60)print(f"{'位置':<6}{'学号':<8}{'姓名':<10}{'成绩'}")print("-" * 60)for i, r in enumerate(sequential_records):    print(f"{i:<6}{r['id']:<8}{r['name']:<10}{r['score']}")# 演示二分查找print("\n--- 二分查找演示 Binary Search Demo ---")target_id = 1012lo, hi = 0, len(sequential_records) - 1steps = 0while lo <= hi:    mid = (lo + hi) // 2    steps += 1    current = sequential_records[mid]    print(f"  第{steps}步: 检查位置 {mid}, 学号={current['id']}  ", end="")    if current['id'] == target_id:        print(f"找到了！")        break    elif current['id'] < target_id:        print(f"太小了，往右找")        lo = mid + 1    else:        print(f"太大了，往左找")        hi = mid - 1print(f"\n在 {len(sequential_records)} 条记录中，只用了 {steps} 步就找到了 ID={target_id}！")print(f"   如果是串行文件，可能需要遍历 {len(sequential_records)//2} 步（平均情况）")

In [ ]:
# 可视化：三种文件组织方式的对比# Visualization: Comparison of three file organization methodsfig, axes = plt.subplots(1, 3, figsize=(16, 5))# --- 串行文件 ---serial_ids = [47, 12, 85, 3, 56, 31]for i, sid in enumerate(serial_ids):    rect = mpatches.FancyBboxPatch((0.1, 0.85 - i * 0.14), 0.8, 0.1,                                     boxstyle="round,pad=0.02", facecolor='#FFB3B3', edgecolor='black')    axes[0].add_patch(rect)    axes[0].text(0.5, 0.9 - i * 0.14, f'ID: {sid}', ha='center', va='center', fontsize=11, fontweight='bold')axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)axes[0].set_title('串行文件 Serial\n(无序)', fontsize=13, fontweight='bold')axes[0].axis('off')# --- 顺序文件 ---seq_ids = sorted(serial_ids)for i, sid in enumerate(seq_ids):    rect = mpatches.FancyBboxPatch((0.1, 0.85 - i * 0.14), 0.8, 0.1,                                     boxstyle="round,pad=0.02", facecolor='#B3FFB3', edgecolor='black')    axes[1].add_patch(rect)    axes[1].text(0.5, 0.9 - i * 0.14, f'ID: {sid}', ha='center', va='center', fontsize=11, fontweight='bold')axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)axes[1].set_title('顺序文件 Sequential\n(按key排序)', fontsize=13, fontweight='bold')axes[1].axis('off')# --- 随机文件 ---hash_slots = [None] * 10for sid in serial_ids:    slot = sid % 10    while hash_slots[slot] is not None:        slot = (slot + 1) % 10    hash_slots[slot] = sidfor i in range(10):    color = '#B3D9FF' if hash_slots[i] is not None else '#F0F0F0'    rect = mpatches.FancyBboxPatch((0.1, 0.91 - i * 0.095), 0.8, 0.07,                                     boxstyle="round,pad=0.02", facecolor=color, edgecolor='black')    axes[2].add_patch(rect)    label = f'[{i}] ID: {hash_slots[i]}' if hash_slots[i] else f'[{i}] (empty)'    axes[2].text(0.5, 0.945 - i * 0.095, label, ha='center', va='center', fontsize=10, fontweight='bold')axes[2].set_xlim(0, 1); axes[2].set_ylim(0, 1)axes[2].set_title('随机文件 Random\n(哈希定位)', fontsize=13, fontweight='bold')axes[2].axis('off')plt.suptitle('三种文件组织方式对比', fontsize=15, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

### 1.3 随机/直接文件 Random/Direct File**定义**：通过**哈希函数 (hash function)** 将记录的关键字段转换为一个**磁盘地址**，直接定位到记录所在的位置。**生活类比**：想象一个有编号的储物柜。你的学号是 1025，通过一个公式（比如 1025 MOD 100 = 25），你知道你的东西在第 25 号柜子里。不需要逐个检查！**核心公式**：```地址 = hash(key) = key MOD 文件大小```**特点**：- **查找和写入都非常快**（O(1) 平均时间）- 需要处理**哈希冲突 (hash collision)**——两个不同的 key 算出同一个地址- 文件中会有**空闲空间浪费**（通常需要比实际记录数多 20%-30% 的空间）- **不适合按顺序遍历**所有记录- 适合**需要快速随机访问**的场景（如银行账户查询、学生信息系统）**处理哈希冲突的方法**：| 方法 | 英文 | 原理 ||------|------|------|| 开放寻址 | Open Addressing | 如果位置被占，就往后找下一个空位 || 链地址法 | Chaining | 每个位置维护一个链表，冲突的记录加入链表 || 溢出区 | Overflow Area | 冲突的记录放到一个专门的溢出区域 |

In [ ]:
# 演示：哈希算法和冲突处理（开放寻址 - 线性探测）# Demo: Hashing Algorithm and Collision Handling (Open Addressing - Linear Probing)class RandomAccessFile:    """模拟随机访问文件 - 使用开放寻址法处理冲突"""    def __init__(self, size=10):        self.size = size        self.slots = [None] * size    def hash_function(self, key):        return key % self.size    def insert(self, key, value):        addr = self.hash_function(key)        original_addr = addr        probes = 0        while self.slots[addr] is not None:            probes += 1            addr = (addr + 1) % self.size            if addr == original_addr:                print(f"  文件已满！无法插入 key={key}")                return        self.slots[addr] = (key, value)        collision = "冲突!" if probes > 0 else "直接插入"        print(f"  插入 key={key:>4}: hash={original_addr} -> 实际地址={addr} ({collision}, 探测{probes}次)")    def search(self, key):        addr = self.hash_function(key)        original_addr = addr        probes = 0        while self.slots[addr] is not None:            if self.slots[addr][0] == key:                return addr, probes            probes += 1            addr = (addr + 1) % self.size            if addr == original_addr:                break        return None, probes# 创建随机文件print("=" * 60)print("随机文件演示 - 哈希函数: key MOD 10")print("=" * 60)raf = RandomAccessFile(10)keys_to_insert = [1025, 3367, 2114, 5542, 7735, 9912, 4465]print("\n【插入过程】")for k in keys_to_insert:    raf.insert(k, f"Record_{k}")print("\n【文件内容】")print(f"{'地址':<6}{'Key':<8}{'Value'}")print("-" * 40)for i, slot in enumerate(raf.slots):    if slot:        print(f"{i:<6}{slot[0]:<8}{slot[1]}")    else:        print(f"{i:<6}{'---':<8}(空 empty)")# 查找演示print("\n【查找演示】")for search_key in [2114, 5542, 9999]:    addr, probes = raf.search(search_key)    if addr is not None:        print(f"  查找 key={search_key}: 找到在地址 {addr} (探测 {probes} 次)")    else:        print(f"  查找 key={search_key}: 未找到 (探测 {probes} 次)")

In [ ]:
# 可视化：哈希冲突处理 - 开放寻址法# Visualization: Hash Collision Resolution - Open Addressingfig, ax = plt.subplots(figsize=(14, 7))slot_height = 0.6slot_width = 1.2start_x = 0.5start_y = 6.0for i in range(10):    y = start_y - i * slot_height    color = '#B3D9FF' if raf.slots[i] else '#F0F0F0'    rect = mpatches.FancyBboxPatch((start_x, y), slot_width * 2.5, slot_height * 0.8,                                     boxstyle="round,pad=0.05", facecolor=color, edgecolor='black', linewidth=1.5)    ax.add_patch(rect)    ax.text(start_x - 0.3, y + slot_height * 0.4, f'[{i}]', ha='center', va='center',            fontsize=12, fontweight='bold', color='#333')    if raf.slots[i]:        ax.text(start_x + slot_width * 1.25, y + slot_height * 0.4,                f'Key: {raf.slots[i][0]}', ha='center', va='center', fontsize=11, fontweight='bold')    else:        ax.text(start_x + slot_width * 1.25, y + slot_height * 0.4,                '(empty)', ha='center', va='center', fontsize=11, color='gray')examples_x = 5.5ax.text(examples_x, 6.5, '哈希计算过程 Hash Computation', fontsize=13, fontweight='bold')explanations = [    ('1025 MOD 10 = 5 -> addr 5', False),    ('3367 MOD 10 = 7 -> addr 7', False),    ('2114 MOD 10 = 4 -> addr 4', False),    ('5542 MOD 10 = 2 -> addr 2', False),    ('7735 MOD 10 = 5 -> collision! -> addr 6', True),    ('9912 MOD 10 = 2 -> collision! -> addr 3', True),    ('4465 MOD 10 = 5 -> collision x3! -> addr 8', True),]for i, (exp, is_collision) in enumerate(explanations):    color = '#CC0000' if is_collision else '#006600'    ax.text(examples_x, 5.9 - i * 0.55, exp, fontsize=10, color=color, fontfamily='monospace')ax.set_xlim(-0.5, 12)ax.set_ylim(-0.5, 7.5)ax.set_title('随机文件 - 哈希表与冲突处理 (开放寻址/线性探测)', fontsize=14, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()

---## 二、文件访问方式 File Access Methods文件组织是**数据如何存放**，文件访问是**程序如何读取数据**。两者是不同的概念！| 访问方式 | 英文 | 描述 | 适合的文件组织 ||---------|------|------|---------------|| 顺序访问 | Sequential Access | 从头到尾逐条读取 | 串行、顺序 || 随机访问 | Random/Direct Access | 直接跳到指定位置 | 随机 || 索引顺序访问 | Indexed Sequential Access | 通过索引定位再顺序读取 | 索引顺序 |### 2.1 顺序访问 Sequential Access**原理**：从文件开头开始，逐条记录读取，直到找到目标或到达文件末尾。**类比**：听磁带音乐——你必须按顺序听，不能直接跳到第5首歌。```OPENFILE "students.dat" FOR READWHILE NOT EOF("students.dat")    READFILE "students.dat", Record    IF Record.ID = TargetID THEN        OUTPUT Record.Name    ENDIFENDWHILECLOSEFILE "students.dat"```### 2.2 随机/直接访问 Random/Direct Access**原理**：通过计算（哈希函数或记录编号 x 记录大小），直接定位到文件中的某个位置读取。**类比**：听CD音乐——可以直接跳到第5首歌！```Address <- Hash(TargetKey)SEEK "data.dat", AddressREADFILE "data.dat", Record```

In [ ]:
# 实际 Python 演示：顺序访问 vs 随机访问# Actual Python Demo: Sequential vs Random Access using real filesimport tempfile, struct, os# 每条记录：4字节ID + 20字节Name + 4字节Score = 28字节RECORD_SIZE = 28RECORD_FORMAT = '=i20si'  # int, 20-byte string, intstudents = [    (1001, b'Alice',   92), (1002, b'Bob',     85),    (1003, b'Charlie', 78), (1004, b'Diana',   95),    (1005, b'Eve',     88), (1006, b'Frank',   71),    (1007, b'Grace',   93), (1008, b'Hank',    82),]# 写入固定大小记录文件tmpfile = os.path.join(tempfile.gettempdir(), 'students_demo.dat')with open(tmpfile, 'wb') as f:    for sid, name, score in students:        padded_name = name.ljust(20, b'\x00')        f.write(struct.pack(RECORD_FORMAT, sid, padded_name, score))print("=" * 60)print("顺序访问 vs 随机访问 - 查找学号 1006 的记录")print("=" * 60)# 方法1：顺序访问print("\n【方法1: 顺序访问 Sequential Access】")with open(tmpfile, 'rb') as f:    record_num = 0    while True:        data = f.read(RECORD_SIZE)        if not data:            break        record_num += 1        sid, name, score = struct.unpack(RECORD_FORMAT, data)        name = name.rstrip(b'\x00').decode()        found_marker = " <-- 找到了！" if sid == 1006 else ""        print(f"  读取第 {record_num} 条: ID={sid}, Name={name}, Score={score}{found_marker}")        if sid == 1006:            breakprint(f"  -> 共读取了 {record_num} 条记录")# 方法2：随机访问print("\n【方法2: 随机访问 Random/Direct Access】")target_id = 1006record_index = target_id - 1001byte_offset = record_index * RECORD_SIZEprint(f"  计算: 记录索引 = {target_id} - 1001 = {record_index}")print(f"  计算: 字节偏移 = {record_index} x {RECORD_SIZE} = {byte_offset}")with open(tmpfile, 'rb') as f:    f.seek(byte_offset)  # 直接跳转！    data = f.read(RECORD_SIZE)    sid, name, score = struct.unpack(RECORD_FORMAT, data)    name = name.rstrip(b'\x00').decode()    print(f"  直接读取: ID={sid}, Name={name}, Score={score}")print(f"  -> 只需要 1 次磁盘读取操作！")os.remove(tmpfile)

---## 三、索引顺序文件 Indexed Sequential File (ISAM)### 3.1 为什么需要索引？顺序文件可以快速按顺序处理，随机文件可以快速定位单条记录。**能不能两全其美呢？**索引顺序文件 (Indexed Sequential Access Method, ISAM) 就是这样的折中方案：- **主文件**按关键字段顺序存储（可以按顺序遍历）- **索引文件**记录了每个"数据块"的起始关键字和地址（可以快速定位）**类比**：想象一本教科书：- **目录（索引）**：告诉你第3章从第45页开始- **正文（主文件）**：内容按章节顺序排列- 你先查目录找到大概位置，再翻到那一页附近找具体内容### 3.2 索引结构```索引文件 (Index):+----------+---------+| Key范围  | 磁盘地址 |+----------+---------+| 1001-1050| Block 0  || 1051-1100| Block 1  || 1101-1150| Block 2  || 1151-1200| Block 3  |+----------+---------+主文件 (Master File):Block 0: [1001][1005][1012][1020][1035][1048]Block 1: [1052][1060][1075][1082][1091][1099]Block 2: [1102][1115][1123][1130][1142][1148]Block 3: [1155][1167][1178][1185][1192][1200]```### 3.3 查找过程要查找 key=1075：1. **查索引**：1075 在 1051-1100 范围内 -> Block 12. **跳到 Block 1**：在 Block 1 内顺序查找 -> 找到 1075### 3.4 溢出区 Overflow Area当需要在已满的 Block 中插入新记录时，新记录会被放到一个**溢出区 (overflow area)**。原始 Block 中会有一个指针指向溢出区的记录。这就像教科书的附录——正文写不下的内容放到后面的附录里，并在正文中标注"见附录A"。

In [ ]:
# 演示：索引顺序文件的查找过程# Demo: Indexed Sequential File Search Processclass IndexedSequentialFile:    """模拟索引顺序文件"""    def __init__(self, block_size=5):        self.block_size = block_size        self.blocks = []        self.index = []    def build_from_sorted(self, records):        for i in range(0, len(records), self.block_size):            block = records[i:i + self.block_size]            self.blocks.append(block)            self.index.append({                'min_key': block[0]['id'],                'max_key': block[-1]['id'],                'block_num': len(self.blocks) - 1            })    def search(self, target_key):        log = []        log.append("步骤1: 查询索引文件")        target_block = None        for idx_entry in self.index:            log.append(f"  检查索引: [{idx_entry['min_key']}-{idx_entry['max_key']}] -> Block {idx_entry['block_num']}")            if idx_entry['min_key'] <= target_key <= idx_entry['max_key']:                target_block = idx_entry['block_num']                log.append(f"  >> Key {target_key} 在此范围内！定位到 Block {target_block}")                break        if target_block is None:            log.append(f"  Key {target_key} 不在任何索引范围内")            return None, log        log.append(f"\n步骤2: 在 Block {target_block} 内顺序查找")        for record in self.blocks[target_block]:            log.append(f"  检查记录: ID={record['id']}, Name={record['name']}")            if record['id'] == target_key:                log.append(f"  >> 找到目标记录！")                return record, log        log.append(f"  在 Block {target_block} 中未找到")        return None, log# 构建示例数据（已排序）all_records = [    {"id": 1001, "name": "Alice",   "grade": "A"},    {"id": 1005, "name": "Bob",     "grade": "B+"},    {"id": 1012, "name": "Charlie", "grade": "B"},    {"id": 1020, "name": "Diana",   "grade": "A+"},    {"id": 1035, "name": "Eve",     "grade": "A"},    {"id": 1048, "name": "Frank",   "grade": "C+"},    {"id": 1052, "name": "Grace",   "grade": "A"},    {"id": 1060, "name": "Hank",    "grade": "B"},    {"id": 1075, "name": "Ivy",     "grade": "A+"},    {"id": 1082, "name": "Jack",    "grade": "B+"},    {"id": 1091, "name": "Kate",    "grade": "A"},    {"id": 1099, "name": "Leo",     "grade": "B"},    {"id": 1102, "name": "Mia",     "grade": "A+"},    {"id": 1115, "name": "Noah",    "grade": "C"},    {"id": 1123, "name": "Olivia",  "grade": "A"},]isf = IndexedSequentialFile(block_size=5)isf.build_from_sorted(all_records)print("=" * 60)print("索引顺序文件 Indexed Sequential File")print("=" * 60)print("\n【索引文件 Index】")print(f"{'范围':<18}{'Block编号'}")print("-" * 30)for entry in isf.index:    print(f"{entry['min_key']}-{entry['max_key']:<12}Block {entry['block_num']}")print("\n" + "=" * 60)print("搜索 Key = 1075")print("=" * 60)result, log = isf.search(1075)for line in log:    print(line)if result:    print(f"\n结果: ID={result['id']}, Name={result['name']}, Grade={result['grade']}")

In [ ]:
# 可视化：索引顺序文件结构# Visualization: Indexed Sequential File Structurefig, ax = plt.subplots(figsize=(15, 8))ax.text(1.5, 7.5, '索引文件 (Index File)', fontsize=14, fontweight='bold', ha='center',        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFD700', edgecolor='black'))index_entries = [('1001-1035', 'Block 0'), ('1048-1091', 'Block 1'), ('1099-1123', 'Block 2')]for i, (key_range, block) in enumerate(index_entries):    y = 6.5 - i * 0.8    rect = mpatches.FancyBboxPatch((0.3, y), 2.4, 0.55,                                     boxstyle="round,pad=0.05", facecolor='#FFF8DC', edgecolor='#DAA520', linewidth=2)    ax.add_patch(rect)    ax.text(1.5, y + 0.275, f'{key_range} -> {block}', ha='center', va='center', fontsize=11, fontweight='bold')ax.text(7.5, 7.5, '主文件 (Master File) - 数据块', fontsize=14, fontweight='bold', ha='center',        bbox=dict(boxstyle='round,pad=0.3', facecolor='#87CEEB', edgecolor='black'))blocks_data = [    ['1001\nAlice', '1005\nBob', '1012\nCharlie', '1020\nDiana', '1035\nEve'],    ['1048\nFrank', '1052\nGrace', '1060\nHank', '1075\nIvy', '1082\nJack'],    ['1091\nKate', '1099\nLeo', '1102\nMia', '1115\nNoah', '1123\nOlivia'],]for bi, block in enumerate(blocks_data):    y = 6.5 - bi * 1.8    ax.text(4.2, y + 0.5, f'Block {bi}:', fontsize=11, fontweight='bold', color='#333')    for ri, rec in enumerate(block):        x = 4.8 + ri * 1.6        color = '#FF6B6B' if '1075' in rec else '#E0F0FF'        rect = mpatches.FancyBboxPatch((x, y - 0.15), 1.3, 0.9,                                         boxstyle="round,pad=0.05", facecolor=color, edgecolor='#4682B4', linewidth=1.5)        ax.add_patch(rect)        ax.text(x + 0.65, y + 0.3, rec, ha='center', va='center', fontsize=9, fontweight='bold')for i in range(3):    idx_y = 6.5 - i * 0.8 + 0.275    block_y = 6.5 - i * 1.8 + 0.3    ax.annotate('', xy=(4.8, block_y), xytext=(2.7, idx_y),                arrowprops=dict(arrowstyle='->', color='#FF6347', lw=2))ax.text(0.3, 3.3, '搜索 Key=1075:', fontsize=12, fontweight='bold', color='red')ax.text(0.3, 2.8, '1. 查索引 -> 1048-1091 -> Block 1', fontsize=10, color='#CC0000')ax.text(0.3, 2.4, '2. Block 1内顺序查找 -> 找到!', fontsize=10, color='#CC0000')ax.set_xlim(-0.2, 13); ax.set_ylim(1.8, 8.2)ax.set_title('索引顺序文件结构 ISAM Structure', fontsize=15, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()

---## 四、多级索引 Multi-Level Indexes### 为什么需要多级索引？假设你有 **1,000,000 条记录**，每个数据块存 100 条，那么：- 数据块数量 = 10,000 个- 索引文件有 10,000 条索引记录- 搜索索引文件本身也需要时间！**解决方案**：为索引文件再建一层索引——**多级索引**！```Level 2 (Master Index):  [A-M -> Level 1 索引A]  [N-Z -> Level 1 索引B]                              |                        |Level 1 (Cylinder Index): [A-F -> Blk 0-99]        [N-S -> Blk 500-599]                          [G-M -> Blk 100-199]      [T-Z -> Blk 600-699]                               |                        |Level 0 (Data Blocks):   [实际数据记录]            [实际数据记录]```**效果对比**（1,000,000 条记录）：| 方法 | 最坏情况磁盘访问次数 ||------|-------------------|| 串行搜索 | 1,000,000 次 || 单级索引 | ~10,000 + 100 = 10,100 次 || 二级索引 | ~100 + 100 + 100 = 300 次 || 三级索引 | ~10 + 100 + 100 + 100 = 310 次 |这就像：- **一级索引** = 图书馆楼层指示牌（"1楼是文学，2楼是理工"）- **二级索引** = 楼层内的书架标签（"A架是物理，B架是化学"）- **数据** = 具体的书> **注意**：多级索引的层数不宜过多，每多一层索引就多一次磁盘读取。通常 2-3 层就足够了。

In [ ]:
# 可视化：多级索引层次结构# Visualization: Multi-Level Index Hierarchyfig, ax = plt.subplots(figsize=(14, 9))level2_y = 8rect = mpatches.FancyBboxPatch((4, level2_y - 0.3), 6, 0.6,    boxstyle="round,pad=0.1", facecolor='#FF6B6B', edgecolor='black', linewidth=2)ax.add_patch(rect)ax.text(7, level2_y, 'Level 2: 主索引 (Master Index)', ha='center', va='center',        fontsize=13, fontweight='bold', color='white')ax.text(7, level2_y - 0.55, '[1-5000 -> L1_A]  [5001-10000 -> L1_B]', ha='center', fontsize=10)level1_y = 5.8for i, (label, x_pos) in enumerate([("L1_A\n1-2500 -> Blk 0\n2501-5000 -> Blk 1", 2.5),                                      ("L1_B\n5001-7500 -> Blk 2\n7501-10000 -> Blk 3", 11.5)]):    rect = mpatches.FancyBboxPatch((x_pos - 2, level1_y - 0.6), 4, 1.2,        boxstyle="round,pad=0.1", facecolor='#4ECDC4', edgecolor='black', linewidth=2)    ax.add_patch(rect)    ax.text(x_pos, level1_y, label, ha='center', va='center', fontsize=10, fontweight='bold')ax.annotate('', xy=(2.5, level1_y + 0.7), xytext=(5.5, level2_y - 0.35),            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))ax.annotate('', xy=(11.5, level1_y + 0.7), xytext=(8.5, level2_y - 0.35),            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))level0_y = 3.2block_positions = [(0.5, 'Block 0\n1-2500'), (4, 'Block 1\n2501-5000'),                   (8, 'Block 2\n5001-7500'), (11.5, 'Block 3\n7501-10000')]for x, label in block_positions:    rect = mpatches.FancyBboxPatch((x - 1, level0_y - 0.5), 2.5, 1,        boxstyle="round,pad=0.1", facecolor='#45B7D1', edgecolor='black', linewidth=2)    ax.add_patch(rect)    ax.text(x + 0.25, level0_y, label, ha='center', va='center', fontsize=10, fontweight='bold', color='white')for src_x, dst_positions in [(2.5, [0.5, 4]), (11.5, [8, 11.5])]:    for dst_x in dst_positions:        ax.annotate('', xy=(dst_x + 0.25, level0_y + 0.55), xytext=(src_x, level1_y - 0.7),                    arrowprops=dict(arrowstyle='->', lw=1.5, color='#666'))ax.text(0, 9, '查找 Key=6200 的路径:', fontsize=12, fontweight='bold', color='red')ax.text(0, 8.5, '(1) 主索引 -> 5001-10000 -> L1_B', fontsize=10, color='#CC0000')ax.text(0, 8.1, '(2) 柱面索引 L1_B -> 5001-7500 -> Block 2', fontsize=10, color='#CC0000')ax.text(0, 7.7, '(3) Block 2 内顺序查找 -> 找到!', fontsize=10, color='#CC0000')ax.text(0, 7.3, '总共只需 3 次磁盘访问!', fontsize=11, color='#CC0000', fontweight='bold')ax.set_xlim(-2, 14.5); ax.set_ylim(2, 9.5)ax.set_title('多级索引结构 Multi-Level Index Structure', fontsize=15, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()

---## 五、哈希算法深入 Hashing Algorithms Deep Dive### 5.1 哈希函数的设计目标一个好的哈希函数应该：1. **计算快速**：O(1) 时间复杂度2. **分布均匀**：尽量把记录均匀分散到不同地址，减少冲突3. **确定性**：同一个 key 永远得到同一个地址### 5.2 常见哈希函数| 方法 | 公式 | 示例 ||------|------|------|| 取余法 (MOD) | `addr = key MOD N` | `1025 MOD 100 = 25` || 折叠法 (Folding) | 把 key 分段相加 | `123456 -> 12+34+56 = 102` || 平方取中 (Mid-square) | 取 key^2 的中间几位 | `45^2 = 2025 -> 取中间 "02"` |### 5.3 为什么 N 最好是素数？如果 N（哈希表大小）是素数，那么 `key MOD N` 的结果分布更均匀。**反例**：如果 N=10，而很多学号的末位都是 0（如 1010, 1020, 1030...），它们全部映射到地址 0，造成严重冲突！如果 N=11（素数），则 1010 MOD 11 = 9, 1020 MOD 11 = 7, 1030 MOD 11 = 5，分布更均匀。

In [ ]:
# 可视化：不同哈希函数的分布比较# Visualization: Distribution comparison of different hash functionsimport randomrandom.seed(42)keys = [random.randint(1000, 9999) for _ in range(150)]keys += [i * 10 for i in range(100, 150)]fig, axes = plt.subplots(1, 3, figsize=(16, 5))hash_10 = [k % 10 for k in keys]axes[0].hist(hash_10, bins=range(11), edgecolor='black', color='#FF6B6B', alpha=0.8, align='left')axes[0].set_title('key MOD 10 (非素数)', fontsize=13, fontweight='bold')axes[0].set_xlabel('哈希地址'); axes[0].set_ylabel('记录数量')axes[0].set_xticks(range(10))hash_11 = [k % 11 for k in keys]axes[1].hist(hash_11, bins=range(12), edgecolor='black', color='#4ECDC4', alpha=0.8, align='left')axes[1].set_title('key MOD 11 (素数)', fontsize=13, fontweight='bold')axes[1].set_xlabel('哈希地址'); axes[1].set_ylabel('记录数量')axes[1].set_xticks(range(11))hash_97 = [k % 97 for k in keys]axes[2].hist(hash_97, bins=50, edgecolor='black', color='#45B7D1', alpha=0.8)axes[2].set_title('key MOD 97 (素数, 更大)', fontsize=13, fontweight='bold')axes[2].set_xlabel('哈希地址'); axes[2].set_ylabel('记录数量')plt.suptitle('哈希函数分布比较 - 素数 vs 非素数', fontsize=15, fontweight='bold')plt.tight_layout()plt.show()from collections import Counterfor name, results, n in [('MOD 10', hash_10, 10), ('MOD 11', hash_11, 11), ('MOD 97', hash_97, 97)]:    counts = Counter(results)    values = [counts.get(i, 0) for i in range(n)]    std = np.std(values)    print(f"{name}: 标准差 = {std:.2f} (越小越均匀)")

In [ ]:
# 演示：折叠法哈希函数# Demo: Folding Hash Functiondef folding_hash(key, table_size):    """折叠法哈希：将key的数字分组相加，再取余"""    key_str = str(key)    groups = []    for i in range(0, len(key_str), 2):        group = key_str[i:i+2]        groups.append(int(group))    total = sum(groups)    address = total % table_size    return address, groups, totalprint("=" * 60)print("折叠法哈希 (Folding Hash) 演示")print("=" * 60)print(f"表大小 (Table Size) = 11\n")test_keys = [123456, 789012, 345678, 901234, 567890]for key in test_keys:    addr, groups, total = folding_hash(key, 11)    groups_str = ' + '.join(str(g) for g in groups)    print(f"  Key = {key}")    print(f"    分组: {groups_str} = {total}")    print(f"    地址: {total} MOD 11 = {addr}")    print()

---## 六、真实世界应用 Real-World Applications### 6.1 不同场景下的文件组织选择| 应用场景 | 推荐方式 | 原因 ||---------|---------|------|| 服务器日志 (Server Log) | 串行 | 只需追加写入，按时间顺序记录 || 月度工资处理 | 顺序 | 需要按员工编号顺序批量处理 || 银行账户查询 | 随机 | 需要根据账号快速查找余额 || 大型数据库 | 索引顺序 | 既需要快速查找，也需要按范围遍历 || 电话簿 | 索引顺序 | 按姓名排序，可按首字母索引 || 临时交易数据 | 串行 | 快速收集，稍后批量处理 |### 6.2 文件更新操作对比| 操作 | 串行文件 | 顺序文件 | 随机文件 | 索引顺序 ||------|---------|---------|---------|----------|| 插入 | 快（追加末尾）| 慢（需重写）| 快（哈希定位）| 中等 || 删除 | 慢（需标记）| 慢（需重写）| 快 | 中等 || 修改 | 慢 | 慢（需重写）| 快 | 中等 || 按序遍历 | N/A | 很快 | 不适合 | 快 || 按key查找 | 很慢 | 中等(二分) | 很快 | 快 |

In [ ]:
# 交互演示：三种方式性能对比# Interactive Demo: Performance comparison of three methodsclass StudentFileSystem:    def __init__(self):        self.serial_data = []        self.sequential_data = []        self.hash_table_size = 17        self.hash_table = [None] * self.hash_table_size    def add_student(self, sid, name, score):        record = {'id': sid, 'name': name, 'score': score}        self.serial_data.append(record)        inserted = False        for i, r in enumerate(self.sequential_data):            if r['id'] > sid:                self.sequential_data.insert(i, record)                inserted = True                break        if not inserted:            self.sequential_data.append(record)        addr = sid % self.hash_table_size        while self.hash_table[addr] is not None:            addr = (addr + 1) % self.hash_table_size        self.hash_table[addr] = record    def search_serial(self, target_id):        comparisons = 0        for record in self.serial_data:            comparisons += 1            if record['id'] == target_id:                return record, comparisons        return None, comparisons    def search_sequential(self, target_id):        comparisons = 0        lo, hi = 0, len(self.sequential_data) - 1        while lo <= hi:            comparisons += 1            mid = (lo + hi) // 2            if self.sequential_data[mid]['id'] == target_id:                return self.sequential_data[mid], comparisons            elif self.sequential_data[mid]['id'] < target_id:                lo = mid + 1            else:                hi = mid - 1        return None, comparisons    def search_random(self, target_id):        comparisons = 0        addr = target_id % self.hash_table_size        start_addr = addr        while self.hash_table[addr] is not None:            comparisons += 1            if self.hash_table[addr]['id'] == target_id:                return self.hash_table[addr], comparisons            addr = (addr + 1) % self.hash_table_size            if addr == start_addr:                break        return None, comparisonssfs = StudentFileSystem()students_data = [    (2045, 'Wang Wei', 88), (1023, 'Li Ming', 92), (3078, 'Zhang San', 75),    (1567, 'Chen Si', 95), (2890, 'Liu Wu', 83), (1234, 'Zhao Liu', 79),    (3456, 'Sun Qi', 91), (2111, 'Zhou Ba', 86), (1789, 'Wu Jiu', 77),    (3210, 'Zheng Shi', 90),]for sid, name, score in students_data:    sfs.add_student(sid, name, score)print("=" * 65)print("三种文件方式的查找性能对比")print("=" * 65)print(f"{'目标ID':<10}{'串行(比较次数)':<18}{'顺序(比较次数)':<18}{'随机(比较次数)'}")print("-" * 65)search_targets = [1023, 2111, 3456, 9999]for target in search_targets:    r1, c1 = sfs.search_serial(target)    r2, c2 = sfs.search_sequential(target)    r3, c3 = sfs.search_random(target)    found = 'Found' if r1 else 'N/A'    print(f"{target:<10}{found:>5} {c1:<12}{found:>5} {c2:<12}{found:>5} {c3}")

In [ ]:
# 可视化：性能对比图（不同数据规模）# Visualization: Performance Comparison at Different Scalessizes = [10, 50, 100, 500, 1000, 5000, 10000]serial_avg = [n / 2 for n in sizes]sequential_avg = [np.log2(n) for n in sizes]random_avg = [1.2 for _ in sizes]indexed_avg = [np.log2(n/100) + 50 if n > 100 else np.log2(n) + 1 for n in sizes]fig, ax = plt.subplots(figsize=(12, 6))ax.plot(sizes, serial_avg, 'ro-', linewidth=2, markersize=8, label='串行 Serial (N/2)')ax.plot(sizes, sequential_avg, 'g^-', linewidth=2, markersize=8, label='顺序 Sequential (log N)')ax.plot(sizes, random_avg, 'bs-', linewidth=2, markersize=8, label='随机 Random (O(1))')ax.plot(sizes, indexed_avg, 'mp-', linewidth=2, markersize=8, label='索引顺序 Indexed Sequential')ax.set_xscale('log'); ax.set_yscale('log')ax.set_xlabel('记录数量 Number of Records', fontsize=12)ax.set_ylabel('平均比较次数 Average Comparisons', fontsize=12)ax.set_title('不同文件访问方式的查找效率对比', fontsize=14, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

In [ ]:
# 实际文件操作：顺序文件的更新过程（主文件 + 事务文件 -> 新主文件）# Real File Operations: Sequential File Update (Master + Transaction -> New Master)print("=" * 60)print("顺序文件更新演示 - 银行账户系统")print("=" * 60)master_records = [    {"id": 101, "name": "Alice",  "balance": 5000},    {"id": 105, "name": "Bob",    "balance": 3200},    {"id": 112, "name": "Charlie","balance": 7800},    {"id": 120, "name": "Diana",  "balance": 1500},    {"id": 135, "name": "Eve",    "balance": 6100},]transactions = [    {"action": "UPDATE", "id": 105, "balance_change": +800},    {"action": "INSERT", "id": 110, "name": "Frank", "balance": 2000},    {"action": "DELETE", "id": 120},    {"action": "UPDATE", "id": 135, "balance_change": -500},]print("\n【旧主文件 Old Master File】")print(f"{'ID':<6}{'Name':<10}{'Balance'}")print("-" * 30)for r in master_records:    print(f"{r['id']:<6}{r['name']:<10}{r['balance']}")print("\n【事务文件 Transaction File】")for t in transactions:    if t['action'] == 'UPDATE':        sign = '+' if t['balance_change'] > 0 else ''        print(f"  {t['action']}: ID={t['id']}, 余额变动: {sign}{t['balance_change']}")    elif t['action'] == 'INSERT':        print(f"  {t['action']}: ID={t['id']}, Name={t['name']}, Balance={t['balance']}")    elif t['action'] == 'DELETE':        print(f"  {t['action']}: ID={t['id']}")print("\n--- 合并处理中... ---\n")delete_ids = {t['id'] for t in transactions if t['action'] == 'DELETE'}update_map = {t['id']: t['balance_change'] for t in transactions if t['action'] == 'UPDATE'}inserts = [t for t in transactions if t['action'] == 'INSERT']new_master = []for r in master_records:    if r['id'] in delete_ids:        print(f"  [DELETE] ID={r['id']} ({r['name']})")        continue    new_record = r.copy()    if r['id'] in update_map:        old_balance = r['balance']        new_record['balance'] += update_map[r['id']]        print(f"  [UPDATE] ID={r['id']}, 余额 {old_balance} -> {new_record['balance']}")    new_master.append(new_record)for ins in inserts:    new_record = {"id": ins['id'], "name": ins['name'], "balance": ins['balance']}    new_master.append(new_record)    print(f"  [INSERT] ID={ins['id']} ({ins['name']})")new_master.sort(key=lambda x: x['id'])print("\n【新主文件 New Master File】")print(f"{'ID':<6}{'Name':<10}{'Balance'}")print("-" * 30)for r in new_master:    print(f"{r['id']:<6}{r['name']:<10}{r['balance']}")

---## 七、总结 Summary### 核心概念回顾| 概念 | 要点 ||------|------|| **串行文件 Serial** | 按到达顺序存储，写快读慢 || **顺序文件 Sequential** | 按key排序，适合批量处理，更新需重建 || **随机文件 Random** | 哈希定位，O(1)查找，需处理冲突 || **索引顺序 Indexed Sequential** | 索引+排序数据，兼顾查找和遍历 || **哈希函数** | key MOD N（N最好是素数），需处理冲突 || **多级索引** | 为索引建索引，减少大文件的搜索次数 || **冲突处理** | 开放寻址、链地址法、溢出区 |### Cambridge 考试答题要点1. **描述题**：解释文件组织方式时，要说清楚数据如何存储和如何检索2. **比较题**：用表格对比优缺点，结合具体场景说明3. **计算题**：给定哈希函数和数据，计算地址并处理冲突4. **设计题**：根据需求推荐文件组织方式并解释原因

---## 八、练习 Exercises### 练习1：基础概念请判断以下场景应该使用哪种文件组织方式，并解释原因：1. 一家医院需要存储病人的预约记录，需要根据病人ID快速查找2. 一个天气监测站每分钟记录一次温度数据3. 一家公司每月需要按员工编号顺序处理工资单4. 一个在线购物网站需要根据订单号快速查找订单，同时也需要按日期范围查询### 练习2：哈希计算已知哈希函数为 `address = key MOD 11`，文件有 11 个槽位（0-10）。按顺序插入以下 key：**25, 36, 14, 47, 58, 69**使用**线性探测**处理冲突，完成下表：| Key | hash(key) | 实际地址 | 是否冲突 ||-----|-----------|---------|----------|| 25  | ?         | ?       | ?        || 36  | ?         | ?       | ?        || ... | ...       | ...     | ...      |

In [ ]:
# 练习2：自己动手试试！取消下面的注释来检查你的答案# Exercise 2: Try it yourself! Uncomment below to check your answerdef exercise_hash_check():    table_size = 11    keys = [25, 36, 14, 47, 58, 69]    table = [None] * table_size    print(f"{'Key':<6}{'hash(key)':<12}{'实际地址':<10}{'冲突?'}")    print("-" * 40)    for key in keys:        hash_val = key % table_size        addr = hash_val        collision = False        while table[addr] is not None:            collision = True            addr = (addr + 1) % table_size        table[addr] = key        print(f"{key:<6}{hash_val:<12}{addr:<10}{'是' if collision else '否'}")    print("\n最终哈希表状态:")    for i in range(table_size):        status = f"Key={table[i]}" if table[i] else "(空)"        print(f"  [{i:>2}] {status}")# 取消下面这行的注释来查看答案：# exercise_hash_check()

### 练习3：索引顺序文件一个文件包含以下已排序的记录 key：```Block 0: [105, 112, 118, 125, 131]Block 1: [140, 148, 155, 162, 170]Block 2: [178, 185, 192, 200, 208]Block 3: [215, 222, 230, 238, 245]```**问题**：1. 写出索引文件的内容（每个 Block 的 key 范围和地址）2. 查找 key=192 需要几步？描述查找过程3. 如果要插入 key=150，应该放在哪里？会有什么问题？### 练习4：编程挑战修改下面的代码，实现一个支持**链地址法 (Chaining)** 处理冲突的哈希表：

In [ ]:
# 练习4：链地址法哈希表 - 请完成 TODO 部分# Exercise 4: Chaining Hash Tableclass ChainingHashTable:    def __init__(self, size=7):        self.size = size        self.table = [[] for _ in range(size)]  # 每个槽位是一个列表    def hash_function(self, key):        return key % self.size    def insert(self, key, value):        # TODO: 计算哈希地址，将 (key, value) 添加到对应链表        addr = self.hash_function(key)        self.table[addr].append((key, value))    def search(self, key):        # TODO: 计算哈希地址，在对应链表中查找 key        addr = self.hash_function(key)        for k, v in self.table[addr]:            if k == key:                return v        return None    def display(self):        print("\n链地址法哈希表:")        for i in range(self.size):            chain = ' -> '.join(f'({k},{v})' for k, v in self.table[i])            print(f"  [{i}]: {chain if chain else '(空)'}")# 测试cht = ChainingHashTable(7)test_data = [(15, 'A'), (22, 'B'), (29, 'C'), (36, 'D'), (8, 'E'), (43, 'F')]for k, v in test_data:    cht.insert(k, v)    print(f"插入 ({k}, '{v}') -> 地址 {cht.hash_function(k)}")cht.display()print(f"\n查找 key=29: {cht.search(29)}")print(f"查找 key=99: {cht.search(99)}")

---### 练习5：Cambridge 风格考题**Question**: A company stores employee records in a file. Each record contains:- Employee ID (key field)- Name- Department- SalaryThe company needs to:- Look up individual employees by ID quickly- Produce a monthly report listing all employees in ID order**(a)** Explain why a random file organization alone would NOT be suitable for both requirements. [2 marks]**(b)** Recommend a suitable file organization method. Justify your answer. [3 marks]**(c)** Describe how an indexed sequential file could be used to meet both requirements. [4 marks]**(d)** The hash function used is `address = EmployeeID MOD 997`. Explain why 997 was chosen as the divisor. [2 marks]---**恭喜你完成了文件组织与访问的学习！**下一节我们将学习浮点数的表示与运算。